# Long-Running Tools

Human-in-the-loop with LongRunningFunctionTool.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 12 — Long-Running Tools (Human-in-the-Loop)

Some actions don't finish in one turn — they need a human, or an async job.
Wrap such a function in **`LongRunningFunctionTool`**: it returns an
*intermediate* result (e.g. `{"status": "pending"}`), the agent pauses, and
your app resumes it later with the real result.

```python
from google.adk.tools import LongRunningFunctionTool

approve = LongRunningFunctionTool(func=request_approval)
```

`get_user_choice` is a ready-made built-in long-running tool for asking the
user to pick an option.

## Your task

1. Write `request_approval(amount: int, tool_context) -> dict` returning
   `{"status": "pending", "ticket": ...}`.
2. Wrap it as `approval_tool = LongRunningFunctionTool(func=request_approval)`.
3. Build `root_agent` (name `"approver"`) with `tools=[approval_tool]`.

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["long-running-tools"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.adk.agents import Agent
from google.adk.tools import LongRunningFunctionTool

def request_approval(amount: int, tool_context) -> dict:
    """Requests human approval for a spend amount.

    Args:
        amount: The amount needing approval.
    """
    # TODO: return a pending status dict (don't block).
    return {}

# TODO: wrap request_approval as a LongRunningFunctionTool and build root_agent.
approval_tool = None
root_agent = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, 'Hello!')   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.adk.agents import Agent
from google.adk.tools import LongRunningFunctionTool

def request_approval(amount: int, tool_context) -> dict:
    """Requests human approval for a spend amount.

    Args:
        amount: The amount needing approval.
    """
    return {"status": "pending", "ticket": f"approve-{amount}"}

approval_tool = LongRunningFunctionTool(func=request_approval)

root_agent = Agent(
    name="approver",
    model="gemini-2.5-flash",
    instruction="For any spend, call request_approval and tell the user it is pending.",
    tools=[approval_tool],
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
